### Setup

In [36]:
# !pip install librecommender faker tensorflow torch

In [37]:
# !pip install tensorflow==2.15.0 keras==2.15.0 librecommender==1.5.2

In [38]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from collections import defaultdict
from faker import Faker
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from libreco.data import DatasetPure
from libreco.algorithms import UserCF, ItemCF, ALS, BPR

In [39]:
spark = (SparkSession.builder
    .appName("JobRecommender")
    .config("spark.sql.catalog.nessie.ref", "test")
    .getOrCreate()
)

In [40]:
jobs_df = spark.read.format("iceberg").load("nessie.silver.jobs")
jobs_df.show(5)

+--------------------+--------------+------------+--------------------+--------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+--------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+
|             job_key|processed_date|    platform|         title_clean|location_clean|       company_clean|min_salary|max_salary|currency|salary_type|min_years|max_years|experience_type|education_standard|level_standard|work_form_standard|quantity_normalized|expired_date_norm|         tech_skills|         soft_skills|          skills_all| category_name_final|         description|         requirement|                link|gold_processed|
+--------------------+--------------+------------+--------------------+--------------+--------------------+----------+--

### Lấy dữ liệu việc làm mới nhất

In [6]:
w = Window.partitionBy(
    "job_key"
).orderBy(
    F.col("processed_date").desc()
)

latest_jobs = (
    jobs_df
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)

In [7]:
jobs_df = (
    latest_jobs
    .select(
        "job_key",
        "location_clean",
        "min_salary",
        "max_salary",
        "min_years",
        "max_years",
        "work_form_standard",
        "category_name_final",
        "skills_all"
    )
).toPandas()

jobs_df.head()

,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all
0,00036788d38de1c9d1433e9f36e8ba0de40233bf0ce39c...,Hà Nội,15000000.0,30000000.0,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[React Next JS, Node JS, React JS Java Script,..."
1,001aa442de3e83d685c55fad31227c883c098c0729395f...,Hà Nội,8000000.0,NaN,0.0,0.0,full_time,HOẠT ĐỘNG DỊCH VỤ KHÁC,"[Word, Excel, giao tiếp tiếng Trung, giao tiếp]"
2,002862efca2eef130d31db889706d06c4ae73a2e501642...,Đà Nẵng,50000000.0,NaN,0.0,0.0,full_time,HOẠT ĐỘNG KINH DOANH BẤT ĐỘNG SẢN,[Giao tiếp]
3,003c504f4f710bba48f2b05c5d98971057ddfd7745e04d...,Hồ Chí Minh,NaN,NaN,0.0,1.0,full_time,XÂY DỰNG,"[Auto CAD, triển khai bản vẽ shop drawing CAD ..."
4,003d99eae190f5178c817588b18b722adfbd9dc0aed4cc...,Hà Nội,15000000.0,30000000.0,0.0,0.0,full_time,GIÁO DỤC VÀ ĐÀO TẠO,"[giao tiếp, thuyết phục, chốt sale, Làm việc n..."


### Sinh dữ liệu tương tác người dùng

#### Gom job theo category

In [23]:
jobs_by_category = defaultdict(list)

for _, job in jobs_df.iterrows():
    jobs_by_category[
        job["category_name_final"]
    ].append(job)

#### Sinh user

In [80]:

# =========================
# 1. Clean jobs
# =========================

def clean_skill_list(skills):
    if not isinstance(skills, list):
        return []

    cleaned = []

    for s in skills:
        s = str(s).strip().lower()

        if not s:
            continue

        cleaned.append(s)

    return list(dict.fromkeys(cleaned))


jobs_df = jobs_df.copy()

jobs_df["skills_clean"] = jobs_df["skills_all"].apply(
    clean_skill_list
)

jobs_df = jobs_df[
    jobs_df["skills_clean"].apply(lambda x: len(x) > 0)
].copy()

jobs_df = jobs_df.dropna(
    subset=["category_name_final"]
)

# =========================
# 2. Build category pools
# =========================

jobs_by_category = defaultdict(list)

for _, job in jobs_df.iterrows():
    jobs_by_category[job["category_name_final"]].append(job)

MIN_JOBS_PER_CATEGORY = 300

valid_categories = [
    cate
    for cate, rows in jobs_by_category.items()
    if len(rows) >= MIN_JOBS_PER_CATEGORY
]

print("Valid categories:", len(valid_categories))

# =========================
# 3. Balanced user categories
# =========================

N_USERS = 10000

users_per_category = N_USERS // len(valid_categories)

user_categories = []

for cate in valid_categories:
    user_categories.extend(
        [cate] * users_per_category
    )

while len(user_categories) < N_USERS:
    user_categories.append(
        random.choice(valid_categories)
    )

random.shuffle(user_categories)

# =========================
# 4. Salary fallback stats
# =========================

category_salary_stats = (
    jobs_df
    .groupby("category_name_final")
    .agg({
        "min_salary": "median",
        "max_salary": "median"
    })
)

global_min_salary = jobs_df["min_salary"].median()
global_max_salary = jobs_df["max_salary"].median()

if pd.isna(global_min_salary):
    global_min_salary = 10_000_000

if pd.isna(global_max_salary):
    global_max_salary = 25_000_000

# =========================
# 5. Helper functions
# =========================

def safe_salary(category, col, default_value):
    value = np.nan

    if category in category_salary_stats.index:
        value = category_salary_stats.loc[
            category,
            col
        ]

    if pd.isna(value):
        value = default_value

    return value


def safe_exp(value, default_value):
    if pd.isna(value):
        return default_value

    return value


# =========================
# 6. Generate users
# =========================

users = []

for i in range(N_USERS):

    category = user_categories[i]
    category_jobs = jobs_by_category[category]

    seed_count = min(
        random.randint(2, 3),
        len(category_jobs)
    )

    seed_jobs = random.sample(
        category_jobs,
        seed_count
    )

    primary_job = random.choice(seed_jobs)

    # -----------------------
    # skill profile
    # -----------------------

    primary_skills = (
        primary_job["skills_clean"]
        if isinstance(primary_job["skills_clean"], list)
        else []
    )

    related_skill_pool = set(primary_skills)

    for job in seed_jobs:
        skills = job["skills_clean"]

        if isinstance(skills, list):
            related_skill_pool.update(skills)

    related_skill_pool = list(related_skill_pool)

    if len(primary_skills) > 0:

        min_base = max(
            1,
            int(len(primary_skills) * 0.6)
        )

        max_base = len(primary_skills)

        n_base = random.randint(
            min_base,
            max_base
        )

        base_skills = random.sample(
            primary_skills,
            n_base
        )

    else:
        base_skills = []

    extra_pool = list(
        set(related_skill_pool) - set(base_skills)
    )

    n_extra = random.randint(
        0,
        min(3, len(extra_pool))
    )

    extra_skills = random.sample(
        extra_pool,
        n_extra
    )

    user_skills = list(
        dict.fromkeys(base_skills + extra_skills)
    )

    # fallback nếu vẫn quá ít skill
    if len(user_skills) < 2 and len(related_skill_pool) > 0:
        user_skills = random.sample(
            related_skill_pool,
            min(3, len(related_skill_pool))
        )

    # -----------------------
    # experience profile
    # -----------------------

    min_exp = safe_exp(
        primary_job["min_years"],
        0
    )

    max_exp = safe_exp(
        primary_job["max_years"],
        min_exp + 2
    )

    lower = max(
        0,
        int(min_exp) - 1
    )

    upper = max(
        lower + 1,
        int(max_exp) + 1
    )

    years_exp = random.randint(
        lower,
        upper
    )

    # -----------------------
    # salary profile
    # -----------------------

    base_min_salary = safe_salary(
        category,
        "min_salary",
        global_min_salary
    )

    base_max_salary = safe_salary(
        category,
        "max_salary",
        global_max_salary
    )

    if base_max_salary < base_min_salary:
        base_max_salary = base_min_salary * 1.5

    desired_min_salary = int(
        base_min_salary *
        random.uniform(0.8, 1.0)
    )

    desired_max_salary = int(
        base_max_salary *
        random.uniform(1.0, 1.2)
    )

    if desired_max_salary <= desired_min_salary:
        desired_max_salary = int(
            desired_min_salary *
            random.uniform(1.2, 1.6)
        )

    # -----------------------
    # location
    # -----------------------

    locations = [
        job["location_clean"]
        for job in seed_jobs
        if pd.notna(job["location_clean"])
    ]

    location = (
        random.choice(locations)
        if locations
        else "Không xác định"
    )

    # -----------------------
    # work form
    # -----------------------

    work_forms = [
        job["work_form_standard"]
        for job in seed_jobs
        if pd.notna(job["work_form_standard"])
    ]

    preferred_work_form = (
        random.choice(work_forms)
        if work_forms
        else "unknown"
    )

    users.append({
        "user_id": f"U{i:05d}",
        "location": location,
        "preferred_category": category,
        "years_experience": years_exp,
        "desired_min_salary": desired_min_salary,
        "desired_max_salary": desired_max_salary,
        "preferred_work_form": preferred_work_form,
        "skills": user_skills,
        "seed_job_key": primary_job["job_key"]
    })

users_df = pd.DataFrame(users)

users_df.head()

Valid categories: 15


,user_id,location,preferred_category,years_experience,desired_min_salary,desired_max_salary,preferred_work_form,skills,seed_job_key
0,U00000,Hồ Chí Minh,"CÔNG NGHIỆP CHẾ BIẾN, CHẾ TẠO",2,9681417,18452269,full_time,"[tổ chức huấn luyện an toàn, sơ đồ tổ chức côn...",66422f88a84d909ec84cef1b7a3e3dafd40753de564ee4...
1,U00001,An Giang,GIÁO DỤC VÀ ĐÀO TẠO,0,10998742,20558854,full_time,"[tefl, gmail, gmat, sat, test prep ielts, celt...",148587b750974987d54cb3a825256ad75f28153de42d8d...
2,U00002,Hà Nội,BÁN BUÔN VÀ BÁN LẺ,2,12568040,22457985,full_time,"[tư duy hình ảnh, dựng video, dựng clip tik to...",27f7bbe15c59cb756788506ae0115456a350c5d5215159...
3,U00003,Hà Nội,HOẠT ĐỘNG HÀNH CHÍNH VÀ DỊCH VỤ HỖ TRỢ,2,8788483,15170315,full_time,"[misa amis, word, excel, quy định pháp luật về...",7e1a97e40f4815a1d2b5ccd9c407bdbe3397ff8f90498c...
4,U00004,Khác,BÁN BUÔN VÀ BÁN LẺ,4,10636093,23404653,full_time,"[brand marketing supervisor, làm việc nhóm, đà...",0e893c055a087368547ccf388c13c3b349d6ea474e5c63...


Sinh user sao cho rải đều các category

In [81]:
users_df["preferred_category"].value_counts()

preferred_category
HOẠT ĐỘNG XUẤT BẢN, PHÁT SÓNG, SẢN XUẤT VÀ PHÂN PHỐI NỘI DUNG                                             668
BÁN BUÔN VÀ BÁN LẺ                                                                                        667
GIÁO DỤC VÀ ĐÀO TẠO                                                                                       667
DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG                                                                                667
HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ                                                               667
HOẠT ĐỘNG KINH DOANH BẤT ĐỘNG SẢN                                                                         667
HOẠT ĐỘNG HÀNH CHÍNH VÀ DỊCH VỤ HỖ TRỢ                                                                    667
HOẠT ĐỘNG DỊCH VỤ KHÁC                                                                                    667
Y TẾ VÀ HOẠT ĐỘNG TRỢ GIÚP XÃ HỘI                                                                    

In [82]:
users_spark = spark.createDataFrame(
    users_df
)

users_spark.write.format("iceberg").mode("overwrite").saveAsTable("nessie.silver.users")

In [83]:
users_spark.show(5)

+-------+-----------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+
|user_id|   location|  preferred_category|years_experience|desired_min_salary|desired_max_salary|preferred_work_form|              skills|        seed_job_key|
+-------+-----------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+
| U00000|Hồ Chí Minh|CÔNG NGHIỆP CHẾ B...|               2|           9681417|          18452269|          full_time|[tổ chức huấn luy...|66422f88a84d909ec...|
| U00001|   An Giang| GIÁO DỤC VÀ ĐÀO TẠO|               0|          10998742|          20558854|          full_time|[tefl, gmail, gma...|148587b750974987d...|
| U00002|     Hà Nội|  BÁN BUÔN VÀ BÁN LẺ|               2|          12568040|          22457985|          full_time|[tư duy hình ảnh,...|27f7bbe15c59cb756...|
| U00003|     Hà Nội|HOẠT ĐỘNG HÀNH CH..

#### Sinh interaction

In [41]:
def skill_match_ratio(
    user_skills,
    job_skills
):

    if not user_skills or not job_skills:
        return 0

    user_skills = set(user_skills)
    job_skills = set(job_skills)

    overlap = len(
        user_skills &
        job_skills
    )

    return overlap / len(job_skills)

In [42]:
def calculate_match_score(
    user,
    job
):

    score = 0

    # 0 -> 10
    skill_ratio = skill_match_ratio(
        user["skills"],
        job["skills_all"]
    )

    score += skill_ratio * 10

    # same category
    if (
        user["preferred_category"]
        ==
        job["category_name_final"]
    ):
        score += 3

    # work form
    if (
        user["preferred_work_form"]
        ==
        job["work_form_standard"]
    ):
        score += 1

    # location
    if (
        user["location"]
        ==
        job["location_clean"]
    ):
        score += 1

    # experience
    min_years = (
        job["min_years"]
        if pd.notna(job["min_years"])
        else 0
    )

    max_years = (
        job["max_years"]
        if pd.notna(job["max_years"])
        else min_years + 2
    )

    if (
        min_years
        <=
        user["years_experience"]
        <=
        max_years
    ):
        score += 2

    return score

In [43]:
interactions = []

job_records = jobs_df.to_dict("records")

jobs_by_category = defaultdict(list)

for job in job_records:
    jobs_by_category[job["category_name_final"]].append(job)

rating_map = {
    "view": 1,
    "save": 3,
    "apply": 5
}

for idx, user in users_df.iterrows():

    if idx % 500 == 0:
        print(f"Processing user {idx}/{len(users_df)}")

    user_category = user["preferred_category"]

    same_category_jobs = jobs_by_category.get(
        user_category,
        []
    )

    sampled_jobs = []

    # 90% cùng ngành
    sampled_jobs.extend(
        random.sample(
            same_category_jobs,
            min(45, len(same_category_jobs))
        )
    )

    # 10% ngoài ngành nhưng có liên quan skill
    random_jobs = random.sample(
        job_records,
        min(30, len(job_records))
    )

    related_other_jobs = []

    for job in random_jobs:

        if job["category_name_final"] == user_category:
            continue

        skill_ratio = skill_match_ratio(
            user["skills"],
            job["skills_all"]
        )

        if skill_ratio > 0:
            related_other_jobs.append(job)

    sampled_jobs.extend(
        random.sample(
            related_other_jobs,
            min(5, len(related_other_jobs))
        )
    )

    # remove duplicate
    sampled_jobs = list({
        job["job_key"]: job
        for job in sampled_jobs
    }.values())

    for job in sampled_jobs:

        score = calculate_match_score(
            user,
            job
        )

        if score < 5:
            continue

        # Nếu khác ngành, yêu cầu skill overlap mạnh hơn
        if job["category_name_final"] != user_category:

            ratio = skill_match_ratio(
                user["skills"],
                job["skills_all"]
            )

            if ratio < 0.3:
                continue

        probability = min(
            score / 16,
            0.9
        )

        if random.random() > probability:
            continue

        if score >= 12:
            event = random.choices(
                ["view", "save", "apply"],
                weights=[15, 30, 55]
            )[0]

        elif score >= 9:
            event = random.choices(
                ["view", "save", "apply"],
                weights=[45, 45, 10]
            )[0]

        else:
            event = random.choices(
                ["view", "save"],
                weights=[85, 15]
            )[0]

        interactions.append({
            "user_id": user["user_id"],
            "job_key": job["job_key"],
            "event_type": event,
            "rating": rating_map[event],
            "event_time": datetime.now() - timedelta(
                days=random.randint(0, 180),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59)
            )
        })

interactions_df = pd.DataFrame(interactions)

print(interactions_df.shape)
interactions_df["event_type"].value_counts(normalize=True)

Processing user 0/10000
Processing user 500/10000
Processing user 1000/10000
Processing user 1500/10000
Processing user 2000/10000
Processing user 2500/10000
Processing user 3000/10000
Processing user 3500/10000
Processing user 4000/10000
Processing user 4500/10000
Processing user 5000/10000
Processing user 5500/10000
Processing user 6000/10000
Processing user 6500/10000
Processing user 7000/10000
Processing user 7500/10000
Processing user 8000/10000
Processing user 8500/10000
Processing user 9000/10000
Processing user 9500/10000
(91253, 5)


event_type
view     0.792708
save     0.176696
apply    0.030596
Name: proportion, dtype: float64

In [45]:
interaction_spark = spark.createDataFrame(
    interactions_df
)

interaction_spark.write.format("iceberg").mode("overwrite").saveAsTable("nessie.silver.interactions")

In [89]:
interaction_spark.show(5)

+-------+--------------------+----------+------+--------------------+
|user_id|             job_key|event_type|rating|          event_time|
+-------+--------------------+----------+------+--------------------+
| U00000|b849e31baca466f91...|      view|     1|2025-12-23 12:57:...|
| U00000|fa8c486b2527f7d29...|      view|     1|2026-02-16 23:25:...|
| U00000|2d97feb26fc407fa2...|      view|     1|2026-02-19 22:49:...|
| U00000|f9a6f1bbff4192515...|      view|     1|2026-03-25 14:54:...|
| U00000|4c3a5951bc630cab6...|      save|     3|2026-01-26 05:02:...|
+-------+--------------------+----------+------+--------------------+
only showing top 5 rows



### Lấy dữ liệu từ bảng nếu đã sinh dữ liệu

In [8]:
users_df = spark.read.format("iceberg").load("nessie.silver.users").toPandas()
interactions_df = spark.read.format("iceberg").load("nessie.silver.interactions").toPandas()

### Lấy dữ liệu từ ở ngoài

In [41]:
# Load users and interactions from csv files (if not using Iceberg)
users_df = pd.read_csv("users.csv")
interactions_df = pd.read_csv("interactions.csv")

In [42]:
import json

users_df["skills"] = users_df["skills"].apply(
    json.loads
)

### Tiền xử lý interactions

In [44]:
interaction_cf = (
    interactions_df
    .groupby(["user_id", "job_key"])["rating"]
    .max()
    .reset_index()
)

interaction_cf = interaction_cf.rename(
    columns={
        "user_id": "user",
        "job_key": "item",
        "rating": "label"
    }
)

interaction_cf.head()

,user,item,label
0,U00000,0d3f84988c2af1683fb06fa93dd68366542826bf19d8e2...,3
1,U00000,20dde5a32548b756d63b5788f2758d9abd39877ca6d10c...,1
2,U00000,3ef4ce57083fa7b98574a5f523d92623f4895211396622...,1
3,U00000,4092b95d9f1ad9afc18bc4513633676213485673f69707...,1
4,U00000,4587ed695fe29d3d1b9c4585e460b17d24edd0a24bd866...,3


### Tạo train/test

In [45]:
from sklearn.model_selection import train_test_split

train_parts = []
test_parts = []

for user, group in interaction_cf.groupby("user"):

    if len(group) < 2:
        train_parts.append(group)
        continue

    train_g, test_g = train_test_split(
        group,
        test_size=0.2,
        random_state=42
    )

    train_parts.append(train_g)
    test_parts.append(test_g)

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

test_df = test_df[
    test_df["item"].isin(train_df["item"])
].copy()

In [46]:
train_data, data_info = DatasetPure.build_trainset(
    train_df
)

test_data = DatasetPure.build_testset(
    test_df,
    data_info
)

#### Tạo tập train / test riêng cho BPR

In [47]:
ranking_df = interaction_cf.copy()
ranking_df["label"] = 1

train_rank_df, test_rank_df = train_test_split(
    ranking_df,
    test_size=0.2,
    random_state=42
)

train_rank_data, rank_data_info = DatasetPure.build_trainset(
    train_rank_df
)

test_rank_data = DatasetPure.build_testset(
    test_rank_df,
    rank_data_info
)

### Train mô hình

#### UserCF

In [48]:
usercf = UserCF(
    task="rating",
    data_info=data_info,
    sim_type="cosine",
    k_sim=100
)

usercf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-10 19:13:35
Final block size and num: (9932, 1)
sim_matrix elapsed: 1.131s
sim_matrix, shape: (9932, 9932), num_elements: 481874, density: 0.4885 %


top_k: 100%|██████████| 9932/9932 [00:00<00:00, 20227.53it/s]

#### ItemCF

In [49]:
itemcf = ItemCF(
    task="rating",
    data_info=data_info,
    sim_type="cosine",
    k_sim=100
)

itemcf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-10 19:13:37
Final block size and num: (5545, 6)
sim_matrix elapsed: 6.198s
sim_matrix, shape: (33270, 33270), num_elements: 1123460, density: 0.1015 %


top_k: 100%|██████████| 33270/33270 [00:00<00:00, 72039.29it/s]

#### ALS

In [50]:
als = ALS(
    task="rating",
    data_info=data_info,
    embed_size=32,
    n_epochs=20,
    reg=1e-4,
    seed=42
)

als.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-10 19:13:43
Epoch 1 elapsed: 0.150s
Epoch 2 elapsed: 0.128s
Epoch 3 elapsed: 0.129s
Epoch 4 elapsed: 0.198s
Epoch 5 elapsed: 0.128s
Epoch 6 elapsed: 0.126s
Epoch 7 elapsed: 0.123s
Epoch 8 elapsed: 0.127s
Epoch 9 elapsed: 0.160s
Epoch 10 elapsed: 0.171s
Epoch 11 elapsed: 0.159s
Epoch 12 elapsed: 0.139s
Epoch 13 elapsed: 0.130s
Epoch 14 elapsed: 0.144s
Epoch 15 elapsed: 0.177s
Epoch 16 elapsed: 0.193s
Epoch 17 elapsed: 0.110s
Epoch 18 elapsed: 0.113s
Epoch 19 elapsed: 0.111s
Epoch 20 elapsed: 0.116s


#### BPR

In [51]:
bpr = BPR(
    task="ranking",
    data_info=rank_data_info,
    embed_size=32,
    n_epochs=20,
    lr=0.001,
    reg=1e-4,
    seed=42
)

bpr.fit(
    train_rank_data,
    neg_sampling=True,
    verbose=2
)

Training start time: 2026-06-10 19:13:46


ValueError: Variable embedding/user_embeds_var already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope? Originally defined at:

  File "/opt/conda/lib/python3.11/site-packages/libreco/layers/embedding.py", line 17, in embedding_lookup
  File "/opt/conda/lib/python3.11/site-packages/libreco/algorithms/bpr.py", line 167, in _build_model_tf
  File "/opt/conda/lib/python3.11/site-packages/libreco/algorithms/bpr.py", line 139, in build_model
  File "/opt/conda/lib/python3.11/site-packages/libreco/algorithms/bpr.py", line 253, in fit
  File "/tmp/ipykernel_1553014/1638186687.py", line 11, in <module>


### Recommend cho 1 user

Chọn user U00123

In [18]:
user_id = "U03423"
user_idx = 3423

In [19]:
users_df.loc[user_idx]

user_id                                                           U03423
location                                                          Hà Nội
preferred_category                                    BÁN BUÔN VÀ BÁN LẺ
years_experience                                                       2
desired_min_salary                                              11899114
desired_max_salary                                              22611829
preferred_work_form                                            full_time
skills                 [reception, trình dược viên, sales engineer, t...
seed_job_key           e89eca16f872f69088d1e7d81be464e5ea7cf8f4aa8ee2...
Name: 3423, dtype: object

In [20]:
for skill in users_df.loc[user_idx, "skills"]:
    print(skill)

reception
trình dược viên
sales engineer
tư vấn viên bán hàng
tiếng anh giao tiếp
telesales
tư duy phân tích
nhân viên marketing
tư vấn bán hàng


#### Thử nghiệm trên 4 mô hình

In [21]:
user_recs = usercf.recommend_user(
    user=user_id,
    n_rec=20
)

user_res = user_recs[user_id]
enriched_user = jobs_df[
    jobs_df["job_key"].isin(user_res)
]
enriched_user['method'] = 'user'
enriched_user.head()

/tmp/ipykernel_1553014/850884489.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_user['method'] = 'user'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
244,135b1ab05eea4f810a19a3f2e8f997af384fbfd97451e4...,Hồ Chí Minh,7000000.0,25000000.0,1.0,10.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[nhân viên kinh doanh fmcg, đại diện kinh doan...",user
5472,aa12ccbfca062f1653ba8f93a18e980e69374ad007ddf0...,Hà Nội,10000000.0,15000000.0,1.0,1.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[Content Marketing, viết đa dạng PR, social me...",user
7032,2399be4896076c0fa91ec80a151ee36fe63649122e2927...,Hồ Chí Minh,8000000.0,20000000.0,0.0,0.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[website, giao tiếp, làm việc nhóm, kinh doanh]",user
10333,22641f64218565d4ab3a144f832ad84b6d9d615e2b2715...,Khác,10000000.0,30000000.0,0.0,0.0,full_time,BÁN BUÔN VÀ BÁN LẺ,[sales engineer / business development / nhân ...,user
12573,ce1e4e07184708ba7773b2d47956a4ff7b6e1f90eddf21...,Hà Nội,10000000.0,20000000.0,1.0,1.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[Digital Ad, tư duy dữ liệu, tối ưu hiệu suất,...",user


In [22]:
item_recs = itemcf.recommend_user( 
    user=user_id,
    n_rec=20
)

item_res = item_recs[user_id]
enriched_item = jobs_df[
    jobs_df["job_key"].isin(item_res)
]
enriched_item['method'] = 'item'
enriched_item.head()

/tmp/ipykernel_1553014/3104140752.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_item['method'] = 'item'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
5472,aa12ccbfca062f1653ba8f93a18e980e69374ad007ddf0...,Hà Nội,10000000.0,15000000.0,1.0,1.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[Content Marketing, viết đa dạng PR, social me...",item
7032,2399be4896076c0fa91ec80a151ee36fe63649122e2927...,Hồ Chí Minh,8000000.0,20000000.0,0.0,0.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[website, giao tiếp, làm việc nhóm, kinh doanh]",item
8200,7caa0acd9327606e17acd947b0f13d89bbce2d97f0938f...,Bắc Ninh,15000000.0,NaN,1.0,1.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[SEO, insight]",item
10333,22641f64218565d4ab3a144f832ad84b6d9d615e2b2715...,Khác,10000000.0,30000000.0,0.0,0.0,full_time,BÁN BUÔN VÀ BÁN LẺ,[sales engineer / business development / nhân ...,item
12573,ce1e4e07184708ba7773b2d47956a4ff7b6e1f90eddf21...,Hà Nội,10000000.0,20000000.0,1.0,1.0,full_time,BÁN BUÔN VÀ BÁN LẺ,"[Digital Ad, tư duy dữ liệu, tối ưu hiệu suất,...",item


In [23]:
als_recs = als.recommend_user(
    user=user_id,   
    n_rec=20    
)

als_res = als_recs[user_id]
enriched_als = jobs_df[
    jobs_df["job_key"].isin(als_res)    
]
enriched_als['method'] = 'als'
enriched_als.head()

/tmp/ipykernel_1553014/1723471370.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_als['method'] = 'als'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
126,09c94a136cb2ac28f008d300d9d8840a827f8b059a9667...,Đồng Nai,NaN,NaN,1.0,1.0,full_time,XÂY DỰNG,"[điện dân dụng, thiết bị điện, nguyên lý hoạt ...",als
5060,8aa7aabde55438385b920b1f5ce9afc9ca6ae67504ebd0...,Bình Dương,NaN,NaN,2.0,0.0,full_time,"CÔNG NGHIỆP CHẾ BIẾN, CHẾ TẠO","[senior technical executive, consultant, couns...",als
8141,78e7c026474d199b77e51149aa7335ff637327f001af5a...,Khác,20000000.0,25000000.0,2.0,3.0,full_time,HOẠT ĐỘNG DỊCH VỤ KHÁC,"[giám sát thi công cơ điện, phân tích bản vẽ k...",als
11845,95c232f9f6bd28f4ba5a967b12e1113c61a1888f83f74b...,Khác,15000000.0,18000000.0,3.0,0.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[kỹ sư điện công nghiệp, kỹ sư công trình m e,...",als
22074,ae36f815967014338d4d858837705ce9393bf03758c6da...,Hà Nội,NaN,NaN,1.0,1.0,full_time,"NGHỆ THUẬT, THỂ THAO VÀ GIẢI TRÍ","[Adobe Illustrator, Photoshop, Tư duy Typograp...",als


In [24]:
bpr_recs = bpr.recommend_user(
    user=user_id,
    n_rec=20
)

bpr_res = bpr_recs[user_id]
enriched_bpr = jobs_df[
    jobs_df["job_key"].isin(bpr_res)
]
enriched_bpr['method'] = 'bpr'
enriched_bpr.head()

/tmp/ipykernel_1553014/2677664316.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_bpr['method'] = 'bpr'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,method
5943,ce8ecaa9e2e20b83134c735eed697c59f8ee4fabd63f28...,Hà Nội,30000000.0,40000000.0,5.0,10.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[trưởng phòng\tkhai thác, head of office, TOEI...",bpr
12219,b2ed0aa11db1ab598d97583a5d069edc8e8e74c54d21e5...,Hà Nội,12000000.0,20000000.0,2.0,2.0,full_time,"VẬN TẢI, KHO BÃI","[Kiểm Soát Chi Phí, Logistics, Thủ Tục Hải Qua...",bpr
16021,d93e7360965db3a96cc1e7a34adfcfb19dbba9c56853f0...,Hồ Chí Minh,NaN,NaN,1.0,1.0,full_time,"VẬN TẢI, KHO BÃI","[Import Management, Customs Procedures, Incote...",bpr
17450,49c24ebce607ff03deae7932e8d75e795b0e229337e984...,Hà Nội,NaN,NaN,3.0,3.0,full_time,"VẬN TẢI, KHO BÃI","[Warehouse Management, Transportation, Logisti...",bpr
24371,652af2b9de4f02ec0621e9b2459f851931f745fe22381f...,Hà Nội,350.0,500.0,0.0,0.0,full_time,"VẬN TẢI, KHO BÃI","[Quản Lý Kho, Xuất Nhập Khẩu, Giao Nhận Kho Vậ...",bpr


### Đánh giá mô hình

#### BPR

In [25]:
from libreco.evaluation import evaluate

evaluate(
    model=bpr,
    data=test_rank_data,
    metrics=[
        "precision",
        "recall",
        "ndcg"
    ],
    neg_sampling=True,
    k=10
)

eval_listwise: 100%|██████████| 8041/8041 [02:04<00:00, 64.46it/s] 


{'precision': 0.0016788956597438129,
 'recall': 0.005410515157437182,
 'ndcg': 0.006970194707721787}

#### UserCF

In [52]:
evaluate(
    model=usercf,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise:   0%|          | 0/3 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 2695 and item 3395, proceed with default prediction
No common interaction or similar neighbor for user 2544 and item 30920, proceed with default prediction
No common interaction or similar neighbor for user 9817 and item 3984, proceed with default prediction
No common interaction or similar neighbor for user 4140 and item 4384, proceed with default prediction
No common interaction or similar neighbor for user 656 and item 28739, proceed with default prediction
No common interaction or similar neighbor for user 5758 and item 8978, proceed with default prediction


eval_pointwise: 100%|██████████| 3/3 [00:03<00:00,  1.02s/it]


{'rmse': 1.1981280303816508, 'mae': 0.8462063492900278}

#### ItemCF

In [53]:
evaluate(
    model=itemcf,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise:   0%|          | 0/3 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 2695 and item 3395, proceed with default prediction
No common interaction or similar neighbor for user 2544 and item 30920, proceed with default prediction
No common interaction or similar neighbor for user 9817 and item 3984, proceed with default prediction
No common interaction or similar neighbor for user 4140 and item 4384, proceed with default prediction
No common interaction or similar neighbor for user 656 and item 28739, proceed with default prediction
No common interaction or similar neighbor for user 5758 and item 8978, proceed with default prediction


eval_pointwise: 100%|██████████| 3/3 [00:01<00:00,  2.38it/s]


{'rmse': 1.1921187837951401, 'mae': 0.83860942643853}

#### ALS

In [54]:
evaluate(
    model=als,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise: 100%|██████████| 3/3 [00:00<00:00, 91.63it/s]


{'rmse': 1.2102954, 'mae': 0.59350914}

### Đánh giá chất lượng đề xuất

#### Avg Skill Overlap

In [29]:
def avg_skill_overlap(
    recommended_jobs,
    user_skills
):

    overlaps = []

    for skills in recommended_jobs["skills_all"]:

        overlap = len(
            set(user_skills)
            &
            set(skills)
        )

        overlaps.append(overlap)

    return np.mean(overlaps)

In [30]:
print ("UserCF skill overlap:",
    avg_skill_overlap(
        enriched_user,
        users_df.loc[user_idx, "skills"]
    )
)

print ("ItemCF skill overlap:",
    avg_skill_overlap(
        enriched_item,
        users_df.loc[user_idx, "skills"]
    )
)

print ("ALS skill overlap:",
    avg_skill_overlap(
        enriched_als,
        users_df.loc[user_idx, "skills"]
    )
)

print ("BPR skill overlap:",
    avg_skill_overlap(
        enriched_bpr,
        users_df.loc[user_idx, "skills"]
    )
)

UserCF skill overlap: 0.35
ItemCF skill overlap: 0.15
ALS skill overlap: 0.05
BPR skill overlap: 0.0


#### Avg Category Match

In [31]:
def avg_category_match(
    recommended_jobs,
    user_category
):

    return (
        recommended_jobs[
            "category_name_final"
        ]
        ==
        user_category
    ).mean()

In [32]:
print ("UserCF category m:",
    avg_category_match(
        enriched_user,
        users_df.loc[123, "preferred_category"]
    )
)

UserCF category m: 0.0


#### Avg Experience Match

In [33]:
def avg_exp_match(
    recommended_jobs,
    years_exp
):

    matches = []

    for _, row in recommended_jobs.iterrows():

        min_exp = (
            row["min_years"]
            if pd.notna(row["min_years"])
            else 0
        )

        max_exp = (
            row["max_years"]
            if pd.notna(row["max_years"])
            else 100
        )

        matches.append(
            min_exp <= years_exp <= max_exp
        )

    return np.mean(matches)

### Checking

In [53]:
check_df = (
    interactions_df
    .merge(
        users_df[["user_id", "preferred_category", "skills"]],
        on="user_id",
        how="left"
    )
    .merge(
        jobs_df[["job_key", "category_name_final", "skills_all"]],
        on="job_key",
        how="left"
    )
)

In [54]:
check_df["same_category"] = (
    check_df["preferred_category"]
    ==
    check_df["category_name_final"]
)

check_df["same_category"].mean()

0.9694188591962332

In [55]:
def skill_overlap_count(user_skills, job_skills):
    if not isinstance(user_skills, list) or not isinstance(job_skills, list):
        return 0

    user_set = set(
        str(s).strip().lower()
        for s in user_skills
    )

    job_set = set(
        str(s).strip().lower()
        for s in job_skills
    )

    return len(user_set & job_set)

In [56]:
check_df["skill_overlap"] = check_df.apply(
    lambda row: skill_overlap_count(
        row["skills"],
        row["skills_all"]
    ),
    axis=1
)

In [57]:
(check_df["skill_overlap"] > 0).mean()

0.840386601131717

In [58]:
check_df.groupby("event_type")["skill_overlap"].mean()

event_type
apply    3.686332
save     1.690745
view     1.097425
Name: skill_overlap, dtype: float64

In [59]:
bad_cases = check_df[
    (check_df["same_category"] == False)
    &
    (check_df["skill_overlap"] == 0)
]

bad_cases[
    [
        "user_id",
        "preferred_category",
        "category_name_final",
        "skills",
        "skills_all",
        "event_type",
        "rating"
    ]
].head(20)

,user_id,preferred_category,category_name_final,skills,skills_all,event_type,rating
13367,U01070,HOẠT ĐỘNG DỊCH VỤ KHÁC,BÁN BUÔN VÀ BÁN LẺ,"[ecommerce background, giao tiếp tiếng anh, ma...","[real estate salesperson, bất động sản, bđs, r...",save,3
14869,U01191,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,BÁN BUÔN VÀ BÁN LẺ,"[positive attitude, tiếng việt, food knowledge...",[sales executive - citadines central binh duon...,view,1
15620,U01250,"VẬN TẢI, KHO BÃI","HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[giải quyết vấn đề, làm việc độc lập, kế hoạch...","[analysis, detailed design, coding, creation o...",view,1
17637,U01429,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,BÁN BUÔN VÀ BÁN LẺ,"[written, verbal communication, supply chain, ...",[sales executive - citadines central binh duon...,view,1
18340,U01483,BÁN BUÔN VÀ BÁN LẺ,XÂY DỰNG,"[diễn thuyết, bán hàng offline, livestream, ads]","[sales supervisor, sales executive, area sales...",view,1
25137,U02018,HOẠT ĐỘNG DỊCH VỤ KHÁC,BÁN BUÔN VÀ BÁN LẺ,"[xử lí các vấn đề điện, bảo trì - sửa chữa các...","[trưởng phòng kinh doanh, head of sales, sales...",view,1
25668,U02061,HOẠT ĐỘNG DỊCH VỤ KHÁC,BÁN BUÔN VÀ BÁN LẺ,"[ms office, sắp xếp hàng hóa, linh kiện, phân ...","[chăm sóc khách hàng, Sales Leader BB/ Trưởng ...",view,1
26146,U02103,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"HOẠT ĐỘNG XUẤT BẢN, PHÁT SÓNG, SẢN XUẤT VÀ PHÂ...","[team leader, uld thiết bị chất tải, cargo sal...","[trưởng phòng kinh doanh ota, head of sales, t...",view,1
38147,U03076,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"HOẠT ĐỘNG XUẤT BẢN, PHÁT SÓNG, SẢN XUẤT VÀ PHÂ...","[restaurant supervisor, financial management, ...","[trưởng phòng kinh doanh ota, head of sales, t...",view,1
49314,U03989,BÁN BUÔN VÀ BÁN LẺ,"HOẠT ĐỘNG TÀI CHÍNH, NGÂN HÀNG VÀ BẢO HIỂM","[quản lý dự án, phân tích số liệu, giải quyết ...","[giải pháp thu hồi và quản lý nợ, sản phẩm cho...",view,1


In [60]:
summary = pd.DataFrame({
    "same_category_rate": [check_df["same_category"].mean()],
    "has_skill_overlap_rate": [(check_df["skill_overlap"] > 0).mean()],
    "avg_skill_overlap": [check_df["skill_overlap"].mean()]
})

summary

,same_category_rate,has_skill_overlap_rate,avg_skill_overlap
0,0.969419,0.840387,1.31864
